<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [ ]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122


In [ ]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch

# Load the model
VERBOSE = True
CHOSEN = 'llama'
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"},
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)

def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Error detection

 Each rulebook is edited by inserting errors that are increasingly difficult to spot:
 - level 0: **original**    -> unaltered rulebook
 - level 1: **missing**     -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable**   -> a rule directly contradicts another one
- level 3: **incoherent**   -> a combination of rules hardlocks the game
- level 4: **gamebreaking** -> a coherent but obviously unbalanced mechanic

In [ ]:
prompts = ["""You are an expert game board player.
Examine the rulebook provided by the user.
Proceed with a chain of thought:

- Scan the text linearly and note any statements that conflict with earlier ones.
- For each of the game mechanic check if it is explained.
- Check whether any mechanic could halt the game or give a player an overwhelming advantage.

Report the **first** problem you discover, quoting the relevant line and summarizing its impact.
If nothing stands out, reply “The rules appear consistent.”
"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 5


output_dict = {str(g):{'lvl'+str(l): {str(it): '' for it in range(iterations)} for l in range(5)} for g  in file_names}


for game,prompt,it,lvl in tqdm([(f,p,it,lvl) for f in file_names for p in prompts for it in range(iterations) for lvl in range(5)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/lvl'+str(lvl)+'/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)['choices'][0]['message']['content']
    if(VERBOSE):
        print(out)
    output_dict[game]['lvl'+str(lvl)][str(it)] = out

with open(f'drive/MyDrive/NLP_proj/{CHOSEN}_error_detection.out','w') as f:
    json.dump(dict(output_dict),f)

  1%|          | 1/100 [00:11<19:10, 11.62s/it]

The first problem I discover is the conflict in the rules regarding the "draw train cards" action.

Relevant line:
"In the unlikely event that there are no cards left to draw and there is no discard pile, a player cannot take the “draw train cards” action. Instead they may only take the “claim 1 route” or “draw tickets” actions."

However, earlier in the rulebook, under the "Draw Train Cards" section, it is stated that "When the train deck is exhausted, all discarded train cards are reshuffled into a new deck." This implies that the train deck will be replenished, but it does not specify what happens when the original deck is exhausted, leaving only the discard pile.

Impact:
This conflict creates uncertainty about what happens when the original train deck is exhausted, leaving the discard pile as the only source of cards. If a player is forced to discard the last card in the deck and there is no discard pile, they may be unable to take the "draw train cards" action, which could signif

  2%|▏         | 2/100 [00:22<18:36, 11.39s/it]

After reviewing the rulebook, I've found a potential problem:

**"You may have any number of cards in your hand at any time." (section "Draw Train Cards") vs. "If you are lucky enough to get a locomotive from the top of the deck in a blind draw, it counts as a single card and you may still draw a total of 2 cards that turn." (same section)**

The first statement allows players to have any number of cards in their hand, implying that there is no limit on the number of cards a player can hold. However, the second statement suggests that if a player draws a locomotive from the top of the deck (a blind draw), they can still draw a total of 2 cards that turn, implying that the player's hand is not already full.

This conflict raises questions about how the game handles hand limits and card draws. If a player can have an unlimited number of cards in their hand, but can still draw 2 cards when they draw a locomotive, it creates an unclear situation. For example, if a player draws a locomotive

  3%|▎         | 3/100 [00:32<17:18, 10.70s/it]

After examining the rulebook, I've identified the first potential problem:

"In the unlikely event that there are no cards left to draw and there is no discard pile, a player cannot take the “draw train cards” action. Instead they may only take the “claim 1 route” or “draw tickets” actions."

This statement seems to create a situation where a player cannot take a valid action on their turn, which could lead to a game-halting situation. If the train deck is exhausted and there is no discard pile, it's unclear what happens next, and the rules do not provide a clear resolution. This could potentially create a stalemate or a situation where a player is unable to take their turn, disrupting the flow of the game.

This problem could be exacerbated by the fact that players may be forced to end their turns early, potentially leading to an uneven game state. It's possible that this was intended to be a rare occurrence, but it's still worth noting as a potential issue.


  4%|▍         | 4/100 [00:44<17:52, 11.17s/it]

The rules appear consistent at first glance. However, upon closer inspection, I notice a potential issue in the "Game End" section.

**Problem:** 
The game can end prematurely if a player's train stock gets down to 0, 1, or 2 trains left at the end of their turn, and then they take a final turn. This means that the player will have one last turn to claim routes, draw tickets, or draw train cards, which could potentially allow them to gain an advantage.

**Quoted line:** 
"When a player’s stock of plastic trains gets down to only 0,1, or 2 trains left at the end of their turn, each player, including that player, gets 1 final turn."

**Impact:** 
This rule could allow a player to take a last-ditch effort to score points, potentially gaining an advantage over other players who have already run out of trains. This could lead to an unfair outcome, especially if the player who gets the final turn is able to claim a long route or complete a ticket that would have been impossible for them to a